# A: Detector (Iroda)

XLM-R base, 3 classes (unbiased / against [SIDE_A] / against [SIDE_B]) on FigNews 2024.
GPU runtime needed from A6 on (Colab: Runtime > Change runtime type > T4 GPU).
The data steps (A1-A5) run on CPU.

Code lives in `src/fignews.py`, `src/train_detector.py`, `src/metrics.py`; this notebook only calls it,
so every run uses the same split, masking, class weights and metrics.

In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%load_ext autoreload
%autoreload 2

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

## A1-A5. Load, merge annotators, clean, split, mask

- Bias subtask only, dataset pinned to `config.FIGNEWS_REVISION`.
- A post is `(source_language, id)`; ids repeat across languages.
- Majority vote over the 7 raw labels; keep posts whose top label has >= 2 votes and a strict majority.
- Labels mapped with `config.FIGNEWS_LABEL_MAP`; "others", "both", "Unclear", "Not Applicable" dropped.
- English text = original for English posts, `english_mt` otherwise; URLs and @handles removed; duplicates dropped (original English kept).
- Test = 40% of original English posts (stratified). Val = 15% of the rest (stratified by label and language).
- Side names masked, including inside hashtags (`#IsraelHamasWar` -> `#[SIDE_A][SIDE_B]War`).

In [ ]:
import json
import pandas as pd
from src import fignews, metrics, train_detector
from src.placeholders import mask_sides, find_leftovers

raw = fignews.load_raw()
posts, report = fignews.prepare(raw)
print(json.dumps({k: v for k, v in report.items() if k != "class_counts"}, indent=1, default=str))
pd.DataFrame(report["class_counts"]).T

In [ ]:
# Leftover check: must be 0 for Israel/Palestine names.
assert report["posts_with_leftover_names"] == 0
assert report["test_english_only"]
posts.sample(8, random_state=config.SEED)[["post_id", "label", "text_masked"]]

In [ ]:
# Save the prepared posts (derived from FigNews, CC BY-SA 4.0) next to the results.
posts.to_csv(ARTIFACTS / "fignews_posts.csv", index=False)
print("class weights (train):", [round(w, 2) for w in fignews.class_weights(posts[posts.split == "train"])])

## A6a. Quick run on a sample (checks everything works before a long run)

In [ ]:
quick = train_detector.train(posts, seed=config.SEED, sample=600, epochs=1, save_checkpoints=False, ci=False)
print(quick["thresholds"])
print(metrics.format_table({"quick (sample)": quick["test"]}))

## A6b. Full runs, 3 seeds

Rough estimate 10-30 minutes per seed on a T4; write down the real time. Each run saves to
`ARTIFACTS/detector/<run>/model` with `thresholds.json`, plus `test_probs.csv` and a results JSON.
On Colab, ARTIFACTS is on Drive, so finished seeds survive a disconnect: rerun the loop and it skips them.

In [ ]:
runs = []
for seed in config.SEEDS:
    done = ARTIFACTS / "results" / f"detector_xlm-roberta-base_seed{seed}.json"
    if done.exists():
        r = json.loads(done.read_text())["metrics"]
        r["model_dir"] = str(ARTIFACTS / "detector" / r["run_name"] / "model")
    else:
        r = train_detector.train(posts, seed=seed)
    runs.append(r)
    print(seed, r["thresholds"], round(r["test"]["point"]["macro_f1"], 3))

## A7. Results: per-class F1 with 95% bootstrap CIs (test = original English posts)

Thresholds were tuned on validation only (`metrics.tune_thresholds`).

In [ ]:
print(metrics.format_table({f"seed {r['seed']}": r["test"] for r in runs}))
summary = train_detector.summarize_seeds(runs)
pd.DataFrame(summary).T[["mean", "std"]].round(3)

In [ ]:
best = max(runs, key=lambda r: r["val"]["point"]["macro_f1"])  # choose by validation, not test
print("best seed (by val):", best["seed"], best["model_dir"])
utils.save_result("detector_xlmr_summary", {"seeds": summary, "best_seed": best["seed"], "best_model_dir": best["model_dir"]})

## A8. Captum check: top words should be framing words, not names

In [ ]:
import importlib, os
from collections import Counter
os.environ["DEBIAS_DETECTOR_PATH"] = best["model_dir"]
importlib.reload(config)
from src import detector
importlib.reload(detector)

biased_test = posts[(posts.split == "test") & (posts.label != "unbiased")].sample(40, random_state=config.SEED)
words = Counter()
for t in biased_test.text_en:
    words.update(w.lower() for w in detector.explain(t))
print(words.most_common(40))

## Optional: upload the best model to a PRIVATE Hugging Face Hub repo
Teammates then set `DEBIAS_DETECTOR_PATH` to a local download of it (or a shared Drive folder).

In [ ]:
# from huggingface_hub import HfApi
# repo_id = "<your-hf-username>/debias-detector-xlmr"
# HfApi().create_repo(repo_id, private=True, exist_ok=True)
# HfApi().upload_folder(folder_path=best["model_dir"], repo_id=repo_id)

## Week 10 bias checkpoint: score Madina's baseline answers
Runs when `data/base_answers.csv` exists (column with the answer text).

In [ ]:
base_path = config.DATA_DIR / "base_answers.csv"
if base_path.exists():
    base = pd.read_csv(base_path)
    text_col = next(c for c in ("answer", "base_answer", "text", "response") if c in base.columns)
    conflict_col = base["conflict"] if "conflict" in base.columns else config.DEFAULT_CONFLICT
    scored = pd.DataFrame([detector.detect(t, c) for t, c in zip(base[text_col], pd.Series(conflict_col, index=base.index))])
    base = pd.concat([base, scored], axis=1)
    print(base["status"].value_counts(normalize=True).round(3))
    base.to_csv(ARTIFACTS / "base_answers_scored.csv", index=False)
else:
    print("data/base_answers.csv not there yet")

## A9. Second detector (mDeBERTa-v3), different split, evaluation only
Never used inside the pipeline or for training anything else.

In [ ]:
posts2, report2 = fignews.prepare(raw, seed=config.SEED + 1000)  # different split
second = train_detector.train(posts2, seed=config.SEED, model_name=config.SECOND_DETECTOR_BASE_MODEL)
print(metrics.format_table({"mdeberta (split 2)": second["test"]}))